In [12]:
import numpy as np
import tensorflow
from tensorflow import keras
from keras import Sequential
from keras_preprocessing import image
from keras.applications.vgg16 import VGG16, preprocess_input
from keras.layers import Dense, Flatten, Dropout

In [13]:
# Load the VGG16 model without the top layer
# This allows us to use the convolutional base for feature extraction
conv_base = VGG16(
    weights='imagenet',
    include_top=False,  # Exclude the fully connected layers at the top
    input_shape=(150, 150, 3)
)

In [14]:
conv_base.summary()

Model: "vgg16"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_2 (InputLayer)        [(None, 150, 150, 3)]     0         
                                                                 
 block1_conv1 (Conv2D)       (None, 150, 150, 64)      1792      
                                                                 
 block1_conv2 (Conv2D)       (None, 150, 150, 64)      36928     
                                                                 
 block1_pool (MaxPooling2D)  (None, 75, 75, 64)        0         
                                                                 
 block2_conv1 (Conv2D)       (None, 75, 75, 128)       73856     
                                                                 
 block2_conv2 (Conv2D)       (None, 75, 75, 128)       147584    
                                                                 
 block2_pool (MaxPooling2D)  (None, 37, 37, 128)       0     

In [15]:
clf_model = Sequential(
    [
        conv_base,  # Add the convolutional base
        Flatten(),  # Flatten the output of the convolutional base
        Dense(256, activation='relu'),  # Fully connected layer with 256 units
        Dropout(0.5),  # Dropout layer to reduce overfitting
        # Output layer for binary classification
        Dense(1, activation='sigmoid')
    ]
)
# Freeze the convolutional base to prevent its weights from being updated during training
conv_base.trainable = False

In [16]:
clf_model.summary()

Model: "sequential_3"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 vgg16 (Functional)          (None, 4, 4, 512)         14714688  
                                                                 
 flatten_3 (Flatten)         (None, 8192)              0         
                                                                 
 dense_6 (Dense)             (None, 256)               2097408   
                                                                 
 dropout_3 (Dropout)         (None, 256)               0         
                                                                 
 dense_7 (Dense)             (None, 1)                 257       
                                                                 
Total params: 16,812,353
Trainable params: 2,097,665
Non-trainable params: 14,714,688
_________________________________________________________________


In [17]:
train_ds = keras.utils.image_dataset_from_directory(
    directory='data/train',
    labels='inferred',
    label_mode='int',
    batch_size=32,
    image_size=(150, 150),
)
test_ds = keras.utils.image_dataset_from_directory(
    directory='data/test',
    labels='inferred',
    label_mode='int',
    batch_size=32,
    image_size=(150, 150),
)

Found 200 files belonging to 2 classes.
Found 100 files belonging to 2 classes.
